# Rules Discovery: Extract Rules from Policy Documents

This notebook demonstrates how to use the `RulesDiscovery` module to extract structured rules
from a policy document. This is a **one-time setup process** — the extracted rules are then
used by `RuleValidationService` to validate transaction documents at runtime.

**Pipeline Overview:**
```
Policy Document  →  RulesDiscovery  →  rule_classes config  →  RuleValidationService
  (one-time)          (this notebook)      (stored)              (per transaction)
```

**Inputs:**
- A policy document (PDF or image) — local file or S3
- Discovery configuration (model, prompts)

**Outputs:**
- Structured rules in `rule_classes` format
- Each rule class has: `x-aws-idp-rule-type`, `description`, and `rule_properties`
- Compatible with `RuleValidationService` for downstream transaction validation

## 1. Setup

In [1]:
ROOTDIR = "../.."

# Path to the policy document to extract rules from
POLICY_PDF_PATH = f"{ROOTDIR}/samples/rule-validation/NCCI Medicare Policy Manual.pdf"

In [2]:
# Ensure latest idp_common is installed
%load_ext autoreload
%autoreload 2

%pip uninstall -y idp_common
%pip install -q -e "{ROOTDIR}/lib/idp_common_pkg[dev, all]"
%pip show idp_common | grep -E "Version|Location"

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

Found existing installation: idp_common 0.5.3.dev9
Uninstalling idp_common-0.5.3.dev9:
  Successfully uninstalled idp_common-0.5.3.dev9
Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aws-sam-cli 1.158.0 requires boto3[crt]==1.42.70, but you have boto3 1.42.0 which is incompatible.
aws-sam-cli 1.158.0 requires watchdog==4.0.2, but you have watchdog 6.0.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Version: 0.5.6
Location: /home/ubuntu/Workspace/research/rule_validation_research/genaiic-idp-accelerator/.venv/lib/python3.12/site-packages
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import json
import yaml
import logging
import boto3
from pathlib import Path

# Import the RulesDiscovery module
from idp_common.discovery import RulesDiscovery
from idp_common.config.models import IDPConfig

# Configure logging
logging.basicConfig(level=logging.WARNING)
logging.getLogger('idp_common.discovery').setLevel(logging.INFO)
logging.getLogger('idp_common.bedrock').setLevel(logging.INFO)

print("Libraries imported successfully")

Libraries imported successfully


## 2. Configure Rules Discovery

We create an `IDPConfig` with the rules discovery settings.
The key configuration is under `discovery.rules` — it controls:
- `model`: Which Bedrock model to use
- `agentic.enabled`: Use Strands Agent for self-correcting, schema-enforced extraction
- `system_prompt`: System-level instructions
- `task_prompt`: Custom extraction prompt (optional — defaults are provided)
- `temperature`, `top_p`, `top_k`, `max_tokens`: Model parameters

In [7]:
# Set environment
os.environ['AWS_REGION'] = boto3.session.Session().region_name or 'us-east-1'
os.environ['METRIC_NAMESPACE'] = 'IDP-Modular-Pipeline'

region = os.environ['AWS_REGION']
print(f"AWS Region: {region}")

AWS Region: us-east-1


In [8]:
# Load configuration from YAML (edit config/rules_discovery.yaml to customize)
config_dir = Path("config")
config_files = ["rules_discovery.yaml"]

CONFIG = {}
for config_file in config_files:
    config_path = config_dir / config_file
    if config_path.exists():
        with open(config_path, 'r') as f:
            file_config = yaml.safe_load(f)
            CONFIG.update(file_config)
        print(f"Loaded {config_file}")
    else:
        print(f"WARNING: {config_file} not found in {config_dir}")

print(f"Loaded configuration sections: {list(CONFIG.keys())}")

config = IDPConfig(**CONFIG)

print(f"\nModel: {config.discovery.rules.model}")
print(f"Agentic: {config.discovery.rules.agentic.enabled}")
print(f"Temperature: {config.discovery.rules.temperature}")
print(f"Max Tokens: {config.discovery.rules.max_tokens}")
print(f"Custom prompt: {'Yes' if config.discovery.rules.task_prompt else 'No (using default)'}")

Loaded rules_discovery.yaml
Loaded configuration sections: ['discovery']

Model: us.anthropic.claude-sonnet-4-20250514-v1:0
Agentic: False
Temperature: 0.0
Max Tokens: 64000
Custom prompt: Yes


## 3. Run Rules Discovery

We use `discovery_rules_from_document_local()` which reads the PDF directly from disk
and sends it to Bedrock — no S3 or DynamoDB required.

In [9]:
# Verify the policy document exists
policy_path = Path(POLICY_PDF_PATH)
if not policy_path.exists():
    print(f"ERROR: Policy document not found at {POLICY_PDF_PATH}")
    print(f"Available files in samples/rule-validation/:")
    for f in Path(f"{ROOTDIR}/samples/rule-validation").iterdir():
        print(f"  {f.name}")
else:
    file_size_mb = policy_path.stat().st_size / (1024 * 1024)
    print(f"Policy document: {policy_path.name}")
    print(f"Size: {file_size_mb:.1f} MB")

Policy document: NCCI Medicare Policy Manual.pdf
Size: 0.2 MB


In [10]:
import time

# Initialize RulesDiscovery with local config (no DynamoDB)
discovery = RulesDiscovery(
    input_bucket="unused",  # Not needed for local file
    input_prefix="unused",
    region=region,
    config=config,
)

print("RulesDiscovery initialized")
print(f"Using model: {discovery.rules_config.model}")
print(f"Agentic mode: {discovery.rules_config.agentic.enabled}")
print("\nExtracting rules from policy document...")
print("(This may take a few minutes for large documents)\n")

start_time = time.time()
result = discovery.discovery_rules_from_document_local(POLICY_PDF_PATH)
elapsed = time.time() - start_time

print(f"\nCompleted in {elapsed:.1f} seconds")
print(f"Status: {result['status']}")
print(f"Rule classes extracted: {len(result['rules'])}")

INFO:idp_common.discovery.rules_discovery:Extracting rules from local document: ../../samples/rule-validation/NCCI Medicare Policy Manual.pdf
INFO:idp_common.discovery.rules_discovery:Document size: 170691 bytes
INFO:idp_common.discovery.rules_discovery:Rules discovery using model: us.anthropic.claude-sonnet-4-20250514-v1:0


INFO:idp_common.bedrock.client:Bedrock request attempt 1/7:
INFO:idp_common.bedrock.client:  - model: us.anthropic.claude-sonnet-4-20250514-v1:0
INFO:idp_common.bedrock.client:  - inferenceConfig: {'temperature': 0.0}
INFO:idp_common.bedrock.client:  - system: [{'text': 'You are an expert in analyzing policy documents and extracting business rules, regulations, and compliance requirements. Extract rules as structured, actionable validation statements.'}]
INFO:idp_common.bedrock.client:  - messages: [{'role': 'user', 'content': [{'document': '[document_data]'}, {'text': '<background>\nYou are an expert in regulatory document analysis. Your task is to extract ALL business rules, regulations, compliance requirements, and validation criteria from the attached policy document as a single flat list of rules.\n</background>\n\n<task>\nAnalyze the policy document thoroughly, page by page. For each rule found:\n1. Give the rule a short, descriptive snake_case name (e.g., "bundled_services_not_s

RulesDiscovery initialized
Using model: us.anthropic.claude-sonnet-4-20250514-v1:0
Agentic mode: False

Extracting rules from policy document...
(This may take a few minutes for large documents)



INFO:idp_common.bedrock.client:Bedrock request successful after 1 attempts. Duration: 41.75s
INFO:idp_common.bedrock.client:Token Usage: {'inputTokens': 17454, 'outputTokens': 3608, 'totalTokens': 21062, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0}
INFO:idp_common.discovery.rules_discovery:Successfully extracted 1 rule classes on attempt 1



Completed in 41.9 seconds
Status: SUCCESS
Rule classes extracted: 1


## 4. Explore Extracted Rules

In [11]:
rules = result['rules']

print("=" * 70)
print("EXTRACTED RULE CLASSES")
print("=" * 70)

total_rules = 0
for i, rule_class in enumerate(rules):
    rule_type = rule_class.get('x-aws-idp-rule-type', 'unknown')
    description = rule_class.get('description', 'No description')
    rule_props = rule_class.get('rule_properties', {})
    num_rules = len(rule_props)
    total_rules += num_rules
    
    print(f"\n{'─' * 70}")
    print(f"Rule Class {i+1}: {rule_type}")
    print(f"Description: {description}")
    print(f"Number of rules: {num_rules}")
    print()
    
    for rule_id, rule_def in list(rule_props.items())[:5]:  # Show first 5 rules
        rule_desc = rule_def.get('description', 'N/A')
        page = rule_def.get('page', 'N/A')
        # Truncate long descriptions for display
        if len(rule_desc) > 120:
            rule_desc = rule_desc[:120] + "..."
        print(f"  [{rule_id}] (p.{page}) {rule_desc}")
    
    if num_rules > 5:
        print(f"  ... and {num_rules - 5} more rules")

print(f"\n{'=' * 70}")
print(f"TOTAL: {len(rules)} rule classes, {total_rules} individual rules")
print(f"{'=' * 70}")

EXTRACTED RULE CLASSES

──────────────────────────────────────────────────────────────────────
Rule Class 1: policy_rules
Description: All rules extracted from the policy document
Number of rules: 49

  [report_most_specific_code] (p.V-3) Is the HCPCS/CPT code that describes the procedure performed to the greatest specificity possible reported?
  [report_code_only_if_all_services_performed] (p.V-3) Is a HCPCS/CPT code reported only if all services described by the code are performed?
  [no_multiple_codes_if_single_exists] (p.V-3) Are multiple HCPCS/CPT codes NOT reported if a single HCPCS/CPT code exists that describes the services?
  [no_separate_reporting_of_included_services] (p.V-3) Are services usually performed as part of the procedure as a standard of medical/surgical practice NOT separately report...
  [em_service_with_major_surgery_decision] (p.V-3) Is an E&M service performed on the same date as a major surgical procedure (090-day global period) for the purpose of de...
  ...

In [12]:
# Show the full JSON for the first rule class
if rules:
    print("Full JSON for first rule class:")
    print(json.dumps(rules[0], indent=2))

Full JSON for first rule class:
{
  "x-aws-idp-rule-type": "policy_rules",
  "description": "All rules extracted from the policy document",
  "rule_properties": {
    "report_most_specific_code": {
      "description": "Is the HCPCS/CPT code that describes the procedure performed to the greatest specificity possible reported?",
      "page": "V-3"
    },
    "report_code_only_if_all_services_performed": {
      "description": "Is a HCPCS/CPT code reported only if all services described by the code are performed?",
      "page": "V-3"
    },
    "no_multiple_codes_if_single_exists": {
      "description": "Are multiple HCPCS/CPT codes NOT reported if a single HCPCS/CPT code exists that describes the services?",
      "page": "V-3"
    },
    "no_separate_reporting_of_included_services": {
      "description": "Are services usually performed as part of the procedure as a standard of medical/surgical practice NOT separately reported simply because HCPCS/CPT codes exist for them?",
      "

## 5. Verify Compatibility with RuleValidationService

The extracted rules must be compatible with `RuleValidationService`, which reads:
- `x-aws-idp-rule-type` → rule categories
- `rule_properties[*].description` → individual rule questions to evaluate

In [13]:
print("Verifying RuleValidationService compatibility...\n")

all_valid = True
for i, rule_class in enumerate(rules):
    # Check required field: x-aws-idp-rule-type
    rule_type = rule_class.get('x-aws-idp-rule-type')
    if not rule_type:
        print(f"  ❌ Rule class {i}: Missing x-aws-idp-rule-type")
        all_valid = False
        continue
    
    # Check required field: rule_properties
    rule_props = rule_class.get('rule_properties', {})
    if not rule_props:
        print(f"  ❌ Rule class {i} ({rule_type}): Missing or empty rule_properties")
        all_valid = False
        continue
    
    # Check each rule has a description
    descriptions = [prop.get('description') for prop in rule_props.values() if prop.get('description')]
    
    if len(descriptions) == len(rule_props):
        print(f"  ✅ {rule_type}: {len(rule_props)} rules, all with descriptions")
    else:
        missing = len(rule_props) - len(descriptions)
        print(f"  ⚠️  {rule_type}: {missing} rules missing descriptions")
        all_valid = False

print()
if all_valid:
    print("✅ All rule classes are compatible with RuleValidationService")
else:
    print("⚠️  Some rule classes have issues — review above")

Verifying RuleValidationService compatibility...

  ✅ policy_rules: 49 rules, all with descriptions

✅ All rule classes are compatible with RuleValidationService


## 6. Save Rules

Save the extracted rules for use with the IDP pipeline.

In [14]:
# Create output directory
data_dir = Path(".data/step_rules_discovery")
data_dir.mkdir(parents=True, exist_ok=True)

# Save extracted rules as JSON
rules_path = data_dir / "rule_classes.json"
with open(rules_path, 'w') as f:
    json.dump(rules, f, indent=2)

# Save as YAML config (ready to merge into IDP config)
rules_config_path = data_dir / "rule_classes.yaml"
with open(rules_config_path, 'w') as f:
    yaml.dump({"rule_classes": rules}, f, default_flow_style=False, width=120)

# Save discovery metadata
metadata = {
    "source_document": POLICY_PDF_PATH,
    "model": config.discovery.rules.model,
    "num_rule_classes": len(rules),
    "total_rules": sum(len(rc.get('rule_properties', {})) for rc in rules),
    "rule_types": [rc.get('x-aws-idp-rule-type') for rc in rules],
    "extraction_time_seconds": round(elapsed, 1),
}
metadata_path = data_dir / "discovery_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved rule classes (JSON): {rules_path}")
print(f"Saved rule classes (YAML): {rules_config_path}")
print(f"Saved metadata: {metadata_path}")

Saved rule classes (JSON): .data/step_rules_discovery/rule_classes.json
Saved rule classes (YAML): .data/step_rules_discovery/rule_classes.yaml
Saved metadata: .data/step_rules_discovery/discovery_metadata.json


## 7. Summary

In [15]:
print("=== Rules Discovery Complete ===")
print(f"✅ Policy document analyzed: {Path(POLICY_PDF_PATH).name}")
print(f"✅ Rule classes extracted: {len(rules)}")
print(f"✅ Total individual rules: {sum(len(rc.get('rule_properties', {})) for rc in rules)}")
print(f"✅ Model used: {config.discovery.rules.model}")
print(f"✅ Extraction time: {elapsed:.1f} seconds")
print(f"✅ Data saved to: .data/step_rules_discovery/")
print()
print("Rule types discovered:")
for rc in rules:
    rt = rc.get('x-aws-idp-rule-type', 'unknown')
    n = len(rc.get('rule_properties', {}))
    print(f"  • {rt} ({n} rules)")
print()
print("📌 Next: Use these rules with RuleValidationService to validate transactions")
print("   Load rule_classes.json into your IDP config as the 'rule_classes' field")

=== Rules Discovery Complete ===
✅ Policy document analyzed: NCCI Medicare Policy Manual.pdf
✅ Rule classes extracted: 1
✅ Total individual rules: 49
✅ Model used: us.anthropic.claude-sonnet-4-20250514-v1:0
✅ Extraction time: 41.9 seconds
✅ Data saved to: .data/step_rules_discovery/

Rule types discovered:
  • policy_rules (49 rules)

📌 Next: Use these rules with RuleValidationService to validate transactions
   Load rule_classes.json into your IDP config as the 'rule_classes' field
